# Batch Parser Evaluation

Evaluation of `assessment_processor.py` over the configured batch of DOCX assessment documents.

## Setup

Imports, parser setup, batch discovery, and shared evaluation helpers.

In [1]:
from collections import Counter
from pathlib import Path
import re
import statistics
import sys
import zipfile
import xml.etree.ElementTree as ET

from docx import Document
from docx.table import Table
from docx.text.paragraph import Paragraph


def find_preprocessing_dir(start: Path) -> Path:
    for candidate in [start, *start.parents]:
        direct = candidate / "assessment_processor.py"
        nested = candidate / "preprocessing" / "assessment_processor.py"
        if direct.exists():
            return candidate
        if nested.exists():
            return nested.parent
    raise FileNotFoundError("Could not locate preprocessing/assessment_processor.py")


preprocessing_dir = find_preprocessing_dir(Path.cwd().resolve())
if str(preprocessing_dir) not in sys.path:
    sys.path.insert(0, str(preprocessing_dir))

from assessment_processor import AssessmentParser, parse_to_dict

source_parser = AssessmentParser()
BATCH_DOCX_FOLDER = preprocessing_dir / "test_preprocessing" / "word assessment files"

if not BATCH_DOCX_FOLDER.exists():
    raise FileNotFoundError(f"Batch folder not found: {BATCH_DOCX_FOLDER}")

docx_files = sorted(path for path in BATCH_DOCX_FOLDER.glob("*.docx") if not path.name.startswith("~$"))
if not docx_files:
    raise FileNotFoundError(f"No .docx files found in: {BATCH_DOCX_FOLDER}")

print(f"Batch folder: {BATCH_DOCX_FOLDER}")
print(f"DOCX files found: {len(docx_files)}")


Batch folder: /Users/Khalid/Desktop/University/SWP/Documentation/IUCN_RL_Assessment_Reviewer/preprocessing/test_preprocessing/word assessment files
DOCX files found: 101


In [2]:
def score_label(score: float) -> str:
    return "PASS" if score == 1.0 else "FAIL"


def print_score(name: str, score: float, details: str = "") -> None:
    print(f"{score_label(score)}: {name}")
    print(f"Score: {score:.2%}")
    if details:
        print(details)


def tokenize_for_bigrams(text: str):
    return re.findall(r"[A-Za-z0-9]+", text.lower())


def bigram_counts(text_values):
    tokens = tokenize_for_bigrams(" ".join(value for value in text_values if value))
    return Counter(zip(tokens, tokens[1:]))


def flatten_section_paths(node: dict):
    paths = []
    for child in node.get("children", []):
        paths.append(tuple(child.get("path", [])))
        paths.extend(flatten_section_paths(child))
    return paths


def flatten_blocks(node: dict):
    blocks = []
    for block in node.get("blocks", []):
        blocks.append((tuple(node.get("path", [])), block))
    for child in node.get("children", []):
        blocks.extend(flatten_blocks(child))
    return blocks


def collect_docx_plain_text(path: Path):
    doc = Document(str(path))
    values = []
    for item in source_parser._iter_block_items_in_order(doc):
        if isinstance(item, Paragraph):
            text = (item.text or "").strip()
            if text:
                values.append(text)
        elif isinstance(item, Table):
            for row in item.rows:
                for cell in row.cells:
                    for paragraph in cell.paragraphs:
                        text = (paragraph.text or "").strip()
                        if text:
                            values.append(text)
    return values


def collect_json_plain_text_for_bigrams(node: dict):
    values = []
    if node.get("path") and node.get("title"):
        values.append(node["title"])
    for block in node.get("blocks", []):
        if block.get("text"):
            values.append(block["text"])
        for item in block.get("items", []):
            if item.get("text"):
                values.append(item["text"])
        for row in block.get("rows", []):
            values.extend(cell for cell in row if cell)
    for child in node.get("children", []):
        values.extend(collect_json_plain_text_for_bigrams(child))
    return values


def collect_docx_heading_paths(path: Path):
    doc = Document(str(path))
    paths = []
    current_h1 = None
    for item in source_parser._iter_block_items_in_order(doc):
        if not isinstance(item, Paragraph):
            continue
        level = source_parser._heading_level(item)
        title = (item.text or "").strip()
        if not title:
            continue
        if level == 1:
            current_h1 = title
            paths.append((title,))
        elif level == 2:
            path_tuple = (title,) if current_h1 is None else (current_h1, title)
            paths.append(path_tuple)
    return set(paths)


def collect_rich_values(node: dict):
    values = []
    for _, block in flatten_blocks(node):
        if "text_rich" in block:
            values.append(block["text_rich"])
        for item in block.get("items", []):
            if "text_rich" in item:
                values.append(item["text_rich"])
        for row in block.get("rows_rich", []):
            values.extend(row)
    return values


def collect_style_features(node: dict):
    features = set()
    for _, block in flatten_blocks(node):
        if block.get("type") == "style":
            for style_name, snippets in block.get("data", {}).items():
                for snippet in snippets:
                    features.add((style_name, snippet))
    return features

# Raw DOCX/XML helper functions.
W_NS = "http://schemas.openxmlformats.org/wordprocessingml/2006/main"
XML_NS = {"w": W_NS}


def _w_attr(element, name: str):
    return element.get(f"{{{W_NS}}}{name}")


def read_docx_xml(path: Path, member: str):
    try:
        with zipfile.ZipFile(path) as archive:
            return ET.fromstring(archive.read(member))
    except KeyError:
        return None


def docx_body_children_from_xml(path: Path):
    root = read_docx_xml(path, "word/document.xml")
    if root is None:
        return []
    body = root.find("w:body", XML_NS)
    return list(body) if body is not None else []


def local_name(element) -> str:
    return element.tag.rsplit("}", 1)[-1]


def xml_paragraph_text(paragraph_el) -> str:
    parts = []
    for child in paragraph_el.iter():
        tag = local_name(child)
        if tag == "t" and child.text:
            parts.append(child.text)
        elif tag in {"tab", "br", "cr"}:
            parts.append(" ")
    return "".join(parts).replace("\xa0", " ").strip()


def xml_table_cell_text(cell_el) -> str:
    paragraph_texts = [
        xml_paragraph_text(p)
        for p in cell_el.findall(".//w:p", XML_NS)
        if xml_paragraph_text(p)
    ]
    return "\n".join(paragraph_texts).strip()


def docx_style_lookup(path: Path):
    root = read_docx_xml(path, "word/styles.xml")
    styles = {}
    if root is None:
        return styles

    for style in root.findall("w:style", XML_NS):
        style_id = _w_attr(style, "styleId")
        if not style_id:
            continue
        name_el = style.find("w:name", XML_NS)
        based_on_el = style.find("w:basedOn", XML_NS)
        styles[style_id] = {
            "name": (_w_attr(name_el, "val") if name_el is not None else "") or "",
            "based_on": (_w_attr(based_on_el, "val") if based_on_el is not None else None),
        }
    return styles


def normalize_style_label(value: str) -> str:
    return "".join(ch for ch in (value or "").lower() if ch.isalnum())


def xml_heading_level(paragraph_el, styles: dict) -> int | None:
    style_el = paragraph_el.find("./w:pPr/w:pStyle", XML_NS)
    style_id = _w_attr(style_el, "val") if style_el is not None else None
    if not style_id:
        return None

    candidates = []
    seen = set()
    current = style_id
    for _ in range(5):
        if not current or current in seen:
            break
        seen.add(current)
        candidates.append(current)
        info = styles.get(current, {})
        candidates.append(info.get("name", ""))
        current = info.get("based_on")

    normalized = {normalize_style_label(candidate) for candidate in candidates if candidate}
    if normalized & {"heading1", "h1"}:
        return 1
    if normalized & {"heading2", "h2"}:
        return 2
    return None


def collect_docx_plain_text_from_xml(path: Path):
    values = []
    for child in docx_body_children_from_xml(path):
        if local_name(child) == "p":
            text = xml_paragraph_text(child)
            if text:
                values.append(text)
        elif local_name(child) == "tbl":
            for cell in child.findall(".//w:tc", XML_NS):
                text = xml_table_cell_text(cell)
                if text:
                    values.append(text)
    return values


def collect_docx_heading_paths_from_xml(path: Path):
    styles = docx_style_lookup(path)
    paths = []
    current_h1 = None
    for child in docx_body_children_from_xml(path):
        if local_name(child) != "p":
            continue
        level = xml_heading_level(child, styles)
        title = xml_paragraph_text(child)
        if not title:
            continue
        if level == 1:
            current_h1 = title
            paths.append((title,))
        elif level == 2:
            paths.append((title,) if current_h1 is None else (current_h1, title))
    return set(paths)


def xml_has_numbering(paragraph_el) -> bool:
    return paragraph_el.find("./w:pPr/w:numPr", XML_NS) is not None


def collect_docx_structure_counts_from_xml(path: Path):
    styles = docx_style_lookup(path)
    counts = Counter({
        "body_paragraphs": 0,
        "list_items": 0,
        "tables": 0,
        "table_cells_with_text": 0,
    })

    for child in docx_body_children_from_xml(path):
        if local_name(child) == "p":
            if not xml_paragraph_text(child) or xml_heading_level(child, styles):
                continue
            if xml_has_numbering(child):
                counts["list_items"] += 1
            else:
                counts["body_paragraphs"] += 1
        elif local_name(child) == "tbl":
            counts["tables"] += 1
            for cell in child.findall(".//w:tc", XML_NS):
                if xml_table_cell_text(cell):
                    counts["table_cells_with_text"] += 1
    return counts


def collect_json_structure_counts(node: dict):
    counts = Counter({
        "body_paragraphs": 0,
        "list_items": 0,
        "tables": 0,
        "table_cells_with_text": 0,
    })
    for _, block in flatten_blocks(node):
        if block.get("type") == "paragraph" and block.get("text"):
            counts["body_paragraphs"] += 1
        elif block.get("type") == "list":
            counts["list_items"] += len([item for item in block.get("items", []) if item.get("text")])
        elif block.get("type") == "table":
            counts["tables"] += 1
            for row in block.get("rows", []):
                counts["table_cells_with_text"] += sum(1 for cell in row if cell)
    return counts


def count_similarity(expected: int, actual: int) -> float:
    if expected == actual:
        return 1.0
    if max(expected, actual) == 0:
        return 1.0
    return min(expected, actual) / max(expected, actual)


def collect_docx_comment_facts_from_xml(path: Path):
    comments_root = read_docx_xml(path, "word/comments.xml")
    comment_ids = set()
    comment_text_by_id = {}
    if comments_root is not None:
        for comment in comments_root.findall("w:comment", XML_NS):
            cid = _w_attr(comment, "id")
            if cid is None:
                continue
            comment_ids.add(cid)
            texts = [t.text for t in comment.findall(".//w:t", XML_NS) if t.text]
            comment_text_by_id[cid] = "".join(texts).strip()

    document_root = read_docx_xml(path, "word/document.xml")
    anchored_ids = set()
    if document_root is not None:
        for tag_name in ["commentRangeStart", "commentReference"]:
            for el in document_root.findall(f".//w:{tag_name}", XML_NS):
                cid = _w_attr(el, "id")
                if cid is not None:
                    anchored_ids.add(cid)

    return {"comment_ids": comment_ids, "anchored_ids": anchored_ids, "comment_text_by_id": comment_text_by_id}


def collect_docx_format_features_from_xml(path: Path):
    root = read_docx_xml(path, "word/document.xml")
    features = set()
    if root is None:
        return features

    for run in root.findall(".//w:r", XML_NS):
        if not "".join(t.text for t in run.findall(".//w:t", XML_NS) if t.text).strip():
            continue
        if run.find("./w:rPr/w:b", XML_NS) is not None:
            features.add("bold")
        if run.find("./w:rPr/w:i", XML_NS) is not None:
            features.add("italic")
        vert = run.find("./w:rPr/w:vertAlign", XML_NS)
        vert_value = _w_attr(vert, "val") if vert is not None else None
        if vert_value in {"superscript", "subscript"}:
            features.add(vert_value)
    return features


def collect_json_format_features(node: dict):
    text = "\n".join(collect_rich_values(node))
    features = set()
    if "<b>" in text and "</b>" in text:
        features.add("bold")
    if "<i>" in text and "</i>" in text:
        features.add("italic")
    if "<sup>" in text and "</sup>" in text:
        features.add("superscript")
    if "<sub>" in text and "</sub>" in text:
        features.add("subscript")
    return features


## Parse All Documents

Parse each DOCX file and record any parsing errors.

In [3]:
parsed_documents = {}
parse_errors = []

for path in docx_files:
    try:
        parsed_documents[path] = parse_to_dict(str(path))
    except Exception as exc:
        parse_errors.append((path.name, type(exc).__name__, str(exc)))

parse_success_score = len(parsed_documents) / len(docx_files)
print_score(
    "Batch parse success rate",
    parse_success_score,
    details=f"Parsed successfully: {len(parsed_documents)} / {len(docx_files)}\nErrors: {parse_errors[:10]}",
)


PASS: Batch parse success rate
Score: 100.00%
Parsed successfully: 101 / 101
Errors: []


## Metric 1: Text Bigram Similarity

Weighted precision, recall, and F1 for normalized bigram overlap between source DOCX text and parsed JSON text.

In [4]:
bigram_results = []
total_overlap = 0
total_docx_bigrams = 0
total_json_bigrams = 0

for path, parsed in parsed_documents.items():
    docx_bigrams = bigram_counts(collect_docx_plain_text(path))
    json_bigrams = bigram_counts(collect_json_plain_text_for_bigrams(parsed))
    overlap = sum((docx_bigrams & json_bigrams).values())
    docx_total = sum(docx_bigrams.values())
    json_total = sum(json_bigrams.values())
    recall = overlap / docx_total if docx_total else 1.0
    precision = overlap / json_total if json_total else 1.0
    f1 = 2 * precision * recall / (precision + recall) if (precision + recall) else 0.0
    bigram_results.append((path.name, precision, recall, f1))
    total_overlap += overlap
    total_docx_bigrams += docx_total
    total_json_bigrams += json_total

bigram_precision = total_overlap / total_json_bigrams if total_json_bigrams else 1.0
bigram_recall = total_overlap / total_docx_bigrams if total_docx_bigrams else 1.0
bigram_f1 = 2 * bigram_precision * bigram_recall / (bigram_precision + bigram_recall) if (bigram_precision + bigram_recall) else 0.0
low_bigram_docs = [row for row in bigram_results if row[3] < 1.0][:10]

print_score(
    "Batch text bigram similarity",
    bigram_f1,
    details=(
        f"Weighted precision: {bigram_precision:.2%}\n"
        f"Weighted recall: {bigram_recall:.2%}\n"
        f"DOCX bigrams: {total_docx_bigrams}\n"
        f"JSON bigrams: {total_json_bigrams}\n"
        f"Documents below 100% F1 sample: {low_bigram_docs}"
    ),
)


PASS: Batch text bigram similarity
Score: 100.00%
Weighted precision: 100.00%
Weighted recall: 100.00%
DOCX bigrams: 127811
JSON bigrams: 127811
Documents below 100% F1 sample: []


## Metric 2: Schema Completeness

Presence of required top-level JSON keys.

In [5]:
expected_keys = {"title", "level", "path", "blocks", "children", "comments"}
schema_results = []

for path, parsed in parsed_documents.items():
    actual_keys = set(parsed.keys())
    score = len(expected_keys & actual_keys) / len(expected_keys)
    schema_results.append((path.name, score, sorted(expected_keys - actual_keys)))

schema_score = statistics.mean(score for _, score, _ in schema_results) if schema_results else 0.0
schema_failures = [row for row in schema_results if row[1] < 1.0][:10]
print_score("Batch schema completeness", schema_score, details=f"Documents with missing keys sample: {schema_failures}")

PASS: Batch schema completeness
Score: 100.00%
Documents with missing keys sample: []


## Metric 3: Heading Tree Recall

Recall of Heading 1 and Heading 2 paths in the parsed JSON output.

In [6]:
heading_results = []

for path, parsed in parsed_documents.items():
    expected_heading_paths = collect_docx_heading_paths(path)
    actual_heading_paths = set(flatten_section_paths(parsed))
    score = len(expected_heading_paths & actual_heading_paths) / len(expected_heading_paths) if expected_heading_paths else 1.0
    missing = sorted(expected_heading_paths - actual_heading_paths)
    heading_results.append((path.name, score, missing[:5]))

heading_score = statistics.mean(score for _, score, _ in heading_results) if heading_results else 0.0
heading_failures = [row for row in heading_results if row[1] < 1.0][:10]
print_score("Batch heading tree recall", heading_score, details=f"Documents with missing heading paths sample: {heading_failures}")

PASS: Batch heading tree recall
Score: 100.00%
Documents with missing heading paths sample: []


## Metric 4: Rich-Text Tag Coverage

Coverage of expected rich-text tags across parsed output.

In [7]:
expected_tags = {"<b>", "</b>", "<i>", "</i>", "<sup>", "</sup>"}
all_rich_text = "\n".join(value for parsed in parsed_documents.values() for value in collect_rich_values(parsed))
present_tags = {tag for tag in expected_tags if tag in all_rich_text}
rich_tag_score = len(present_tags) / len(expected_tags)
print_score("Batch rich-text tag coverage", rich_tag_score, details=f"Present tags: {sorted(present_tags)}")

PASS: Batch rich-text tag coverage
Score: 100.00%
Present tags: ['</b>', '</i>', '</sup>', '<b>', '<i>', '<sup>']


## Metric 5: Style Block Presence

Presence of extracted bold or italic style features.

In [8]:
style_presence_results = []

for path, parsed in parsed_documents.items():
    features = collect_style_features(parsed)
    style_presence_results.append((path.name, 1.0 if features else 0.0, len(features)))

style_presence_score = statistics.mean(score for _, score, _ in style_presence_results) if style_presence_results else 0.0
style_missing_docs = [row for row in style_presence_results if row[1] < 1.0][:10]
print_score("Batch style block presence", style_presence_score, details=f"Documents without style features sample: {style_missing_docs}")

PASS: Batch style block presence
Score: 100.00%
Documents without style features sample: []


## Metric 6: Comment Output Structure

Presence of a `comments` list in each parsed document.

In [9]:
comment_structure_results = []

for path, parsed in parsed_documents.items():
    score = 1.0 if isinstance(parsed.get("comments"), list) else 0.0
    comment_structure_results.append((path.name, score, type(parsed.get("comments")).__name__))

comment_structure_score = statistics.mean(score for _, score, _ in comment_structure_results) if comment_structure_results else 0.0
comment_failures = [row for row in comment_structure_results if row[1] < 1.0][:10]
total_comments = sum(len(parsed.get("comments", [])) for parsed in parsed_documents.values() if isinstance(parsed.get("comments"), list))
print_score("Batch comment output structure", comment_structure_score, details=f"Total extracted comments: {total_comments}\nFailures sample: {comment_failures}")

PASS: Batch comment output structure
Score: 100.00%
Total extracted comments: 815
Failures sample: []


## Metric 7: Raw DOCX/XML Text Bigram Similarity

Weighted precision, recall, and F1 for normalized bigram overlap using text extracted from `word/document.xml`.


In [10]:
xml_bigram_results = []
xml_total_overlap = 0
xml_total_docx_bigrams = 0
xml_total_json_bigrams = 0

for path, parsed in parsed_documents.items():
    docx_bigrams = bigram_counts(collect_docx_plain_text_from_xml(path))
    json_bigrams = bigram_counts(collect_json_plain_text_for_bigrams(parsed))
    overlap = sum((docx_bigrams & json_bigrams).values())
    docx_total = sum(docx_bigrams.values())
    json_total = sum(json_bigrams.values())
    recall = overlap / docx_total if docx_total else 1.0
    precision = overlap / json_total if json_total else 1.0
    f1 = 2 * precision * recall / (precision + recall) if (precision + recall) else 0.0
    xml_bigram_results.append((path.name, precision, recall, f1))
    xml_total_overlap += overlap
    xml_total_docx_bigrams += docx_total
    xml_total_json_bigrams += json_total

xml_bigram_precision = xml_total_overlap / xml_total_json_bigrams if xml_total_json_bigrams else 1.0
xml_bigram_recall = xml_total_overlap / xml_total_docx_bigrams if xml_total_docx_bigrams else 1.0
xml_bigram_f1 = 2 * xml_bigram_precision * xml_bigram_recall / (xml_bigram_precision + xml_bigram_recall) if (xml_bigram_precision + xml_bigram_recall) else 0.0
xml_low_bigram_docs = [row for row in xml_bigram_results if row[3] < 0.98][:10]

print_score(
    "Raw DOCX/XML text bigram similarity",
    xml_bigram_f1,
    details=(
        f"Weighted precision: {xml_bigram_precision:.2%}\n"
        f"Weighted recall: {xml_bigram_recall:.2%}\n"
        f"Raw DOCX/XML bigrams: {xml_total_docx_bigrams}\n"
        f"JSON bigrams: {xml_total_json_bigrams}\n"
        f"Documents below 98% F1 sample: {xml_low_bigram_docs}"
    ),
)


FAIL: Raw DOCX/XML text bigram similarity
Score: 94.64%
Weighted precision: 97.41%
Weighted recall: 92.03%
Raw DOCX/XML bigrams: 135276
JSON bigrams: 127811
Documents below 98% F1 sample: [('Acrocarpus_fraxinifolius_JP (2).docx', 0.9647058823529412, 0.9196261682242991, 0.9416267942583733), ('Adinandra oblonga Craib_JP.docx', 0.9781328847771237, 0.9289137380191693, 0.9528881605899221), ('Afzelia xylocarpa (Kurz) Craib_JP.docx', 0.9717634523175279, 0.8807339449541285, 0.9240121580547112), ('Alangium_sempervirens_JP.docx', 0.9662363455809335, 0.9337811900191939, 0.9497315763787214), ('Allophylus sootepensis Craib_JP.docx', 0.9542857142857143, 0.9095860566448801, 0.9313998884551031), ('Archidendron_bigeminum_JP.docx', 0.9782608695652174, 0.9395973154362416, 0.9585393685812096), ('Ardisia ionantha K.Larsen & C.M.Hu_JP.docx', 0.9763458401305057, 0.9061317183951552, 0.939929328621908), ('Ardisia multipunctata H.R.Fletcher_JP.docx', 0.9638888888888889, 0.8685857321652065, 0.9137590520079), ('A

## Metric 8: Raw DOCX/XML Heading Tree Recall

Recall of Heading 1 and Heading 2 paths detected from raw DOCX style XML.


In [11]:
xml_heading_results = []

for path, parsed in parsed_documents.items():
    expected_heading_paths = collect_docx_heading_paths_from_xml(path)
    actual_heading_paths = set(flatten_section_paths(parsed))
    score = len(expected_heading_paths & actual_heading_paths) / len(expected_heading_paths) if expected_heading_paths else 1.0
    missing = sorted(expected_heading_paths - actual_heading_paths)
    extra = sorted(actual_heading_paths - expected_heading_paths)
    xml_heading_results.append((path.name, score, missing[:5], extra[:5]))

xml_heading_score = statistics.mean(score for _, score, _, _ in xml_heading_results) if xml_heading_results else 0.0
xml_heading_failures = [row for row in xml_heading_results if row[1] < 1.0][:10]
print_score(
    "Raw DOCX/XML heading tree recall",
    xml_heading_score,
    details=f"Documents with missing heading paths sample: {xml_heading_failures}",
)


FAIL: Raw DOCX/XML heading tree recall
Score: 99.91%
Documents with missing heading paths sample: [('Psydrax calcicola (Craib) A.P.Davis_JP.docx', 0.967741935483871, [('Conservation', 'Important Conservation Actions NeededA')], [('Conservation', 'Important Conservation Actions Needed')]), ('Tribounia grandiflora_JP.docx', 0.9393939393939394, [('RThreats',), ('RThreats', 'Threats Classification Scheme')], [('Threats',), ('Threats', 'Threats Classification Scheme')])]


## Metric 9: Raw DOCX/XML Structure Count Consistency

Consistency between raw DOCX XML structure counts and parsed JSON block counts.


In [12]:
structure_results = []
count_keys = ["body_paragraphs", "list_items", "tables", "table_cells_with_text"]

for path, parsed in parsed_documents.items():
    expected_counts = collect_docx_structure_counts_from_xml(path)
    actual_counts = collect_json_structure_counts(parsed)
    per_count_scores = {key: count_similarity(expected_counts[key], actual_counts[key]) for key in count_keys}
    score = statistics.mean(per_count_scores.values())
    mismatches = {
        key: {"expected": expected_counts[key], "actual": actual_counts[key], "score": per_count_scores[key]}
        for key in count_keys
        if expected_counts[key] != actual_counts[key]
    }
    structure_results.append((path.name, score, mismatches))

structure_count_score = statistics.mean(score for _, score, _ in structure_results) if structure_results else 0.0
structure_count_failures = [row for row in structure_results if row[1] < 1.0][:10]
print_score(
    "Raw DOCX/XML structure count consistency",
    structure_count_score,
    details=f"Documents with count mismatches sample: {structure_count_failures}",
)


FAIL: Raw DOCX/XML structure count consistency
Score: 99.41%
Documents with count mismatches sample: [('Afzelia xylocarpa (Kurz) Craib_JP.docx', 0.9988207547169812, {'table_cells_with_text': {'expected': 212, 'actual': 211, 'score': 0.9952830188679245}}), ('Alangium_sempervirens_JP.docx', 0.9969325153374233, {'table_cells_with_text': {'expected': 161, 'actual': 163, 'score': 0.9877300613496932}}), ('Allophylus sootepensis Craib_JP.docx', 0.9945652173913043, {'table_cells_with_text': {'expected': 90, 'actual': 92, 'score': 0.9782608695652174}}), ('Ardisia ionantha K.Larsen & C.M.Hu_JP.docx', 0.9975247524752475, {'table_cells_with_text': {'expected': 101, 'actual': 100, 'score': 0.9900990099009901}}), ('Ardisia multipunctata H.R.Fletcher_JP.docx', 0.9979674796747967, {'table_cells_with_text': {'expected': 122, 'actual': 123, 'score': 0.991869918699187}}), ('Cibotium barometz draft assessment_JP.docx', 0.9986338797814207, {'table_cells_with_text': {'expected': 183, 'actual': 182, 'score':

## Metric 10: Raw DOCX/XML Comment ID Coverage

Coverage of comment IDs found in raw DOCX XML.


In [13]:
comment_id_results = []

for path, parsed in parsed_documents.items():
    facts = collect_docx_comment_facts_from_xml(path)
    expected_ids = facts["comment_ids"]
    parsed_comments = parsed.get("comments", []) if isinstance(parsed.get("comments"), list) else []
    parsed_by_id = {str(comment.get("id")): comment for comment in parsed_comments if comment.get("id") is not None}
    actual_ids = set(parsed_by_id)

    id_score = len(expected_ids & actual_ids) / len(expected_ids) if expected_ids else (1.0 if not actual_ids else 0.0)
    comment_id_results.append((path.name, id_score, sorted(expected_ids - actual_ids)[:5], sorted(actual_ids - expected_ids)[:5]))

comment_id_score = statistics.mean(score for _, score, _, _ in comment_id_results) if comment_id_results else 0.0
comment_id_failures = [row for row in comment_id_results if row[1] < 1.0][:10]

print_score(
    "Raw DOCX/XML comment ID coverage",
    comment_id_score,
    details=f"Documents with missing/extra comment IDs sample: {comment_id_failures}",
)


PASS: Raw DOCX/XML comment ID coverage
Score: 100.00%
Documents with missing/extra comment IDs sample: []


## Metric 11: Raw DOCX/XML Rich-Text Feature Recall

Recall of bold, italic, superscript, and subscript features detected from raw DOCX XML.


In [14]:
format_feature_results = []

for path, parsed in parsed_documents.items():
    expected_features = collect_docx_format_features_from_xml(path)
    actual_features = collect_json_format_features(parsed)
    score = len(expected_features & actual_features) / len(expected_features) if expected_features else 1.0
    format_feature_results.append((path.name, score, sorted(expected_features), sorted(actual_features), sorted(expected_features - actual_features)))

format_feature_score = statistics.mean(score for _, score, _, _, _ in format_feature_results) if format_feature_results else 0.0
format_feature_failures = [row for row in format_feature_results if row[1] < 1.0][:10]
print_score(
    "Raw DOCX/XML rich-text feature recall",
    format_feature_score,
    details=f"Documents with missing rich-text features sample: {format_feature_failures}",
)


FAIL: Raw DOCX/XML rich-text feature recall
Score: 98.68%
Documents with missing rich-text features sample: [('Adinandra oblonga Craib_JP.docx', 0.6666666666666666, ['bold', 'italic', 'superscript'], ['bold', 'italic'], ['superscript']), ('Allophylus sootepensis Craib_JP.docx', 0.6666666666666666, ['bold', 'italic', 'superscript'], ['bold', 'italic'], ['superscript']), ('Microchirita hemratii_JP.docx', 0.6666666666666666, ['bold', 'italic', 'superscript'], ['bold', 'italic'], ['superscript']), ('Viburnum_cotinifolium_JP.docx', 0.6666666666666666, ['bold', 'italic', 'superscript'], ['bold', 'italic'], ['superscript'])]


## Overall Batch Evaluation Score

Aggregate scores for batch-level parser performance.

In [15]:
metric_scores = {
    "parse_success_rate": parse_success_score,
    "docx_api_text_bigram_similarity": bigram_f1,
    "schema_completeness": schema_score,
    "docx_api_heading_tree_recall": heading_score,
    "rich_text_tag_coverage": rich_tag_score,
    "style_block_presence": style_presence_score,
    "comment_output_structure": comment_structure_score,
    "raw_xml_text_bigram_similarity": xml_bigram_f1,
    "raw_xml_heading_tree_recall": xml_heading_score,
    "raw_xml_structure_count_consistency": structure_count_score,
    "raw_xml_comment_id_coverage": comment_id_score,
    "raw_xml_rich_text_feature_recall": format_feature_score,
}

raw_xml_metric_scores = {
    "raw_xml_text_bigram_similarity": xml_bigram_f1,
    "raw_xml_heading_tree_recall": xml_heading_score,
    "raw_xml_structure_count_consistency": structure_count_score,
    "raw_xml_comment_id_coverage": comment_id_score,
    "raw_xml_rich_text_feature_recall": format_feature_score,
}

for metric_name, metric_score in metric_scores.items():
    print(f"{metric_name}: {metric_score:.2%}")

print()
raw_xml_score = sum(raw_xml_metric_scores.values()) / len(raw_xml_metric_scores)
print_score("Raw DOCX/XML evaluation score", raw_xml_score)

print()
overall_score = sum(metric_scores.values()) / len(metric_scores)
print_score("Overall batch parser evaluation score", overall_score)


parse_success_rate: 100.00%
docx_api_text_bigram_similarity: 100.00%
schema_completeness: 100.00%
docx_api_heading_tree_recall: 100.00%
rich_text_tag_coverage: 100.00%
style_block_presence: 100.00%
comment_output_structure: 100.00%
raw_xml_text_bigram_similarity: 94.64%
raw_xml_heading_tree_recall: 99.91%
raw_xml_structure_count_consistency: 99.41%
raw_xml_comment_id_coverage: 100.00%
raw_xml_rich_text_feature_recall: 98.68%

FAIL: Raw DOCX/XML evaluation score
Score: 98.53%

FAIL: Overall batch parser evaluation score
Score: 99.39%
